# Module 4.4 — Metaclasses & Class Customisation

### Python Mastery · Track 4: Functional Programming & Metaprogramming

In Python, classes are themselves objects, created by a class-of-classes called a metaclass. Understanding this lets you customise how classes are built, validate them, register them, or inject behaviour. This module covers `type` as a class factory, writing a metaclass, and the lighter hooks `__init_subclass__` and `__set_name__` that handle most real needs.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Observe what happens at class-creation time versus instance-creation time.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Explain that classes are objects and `type` is their metaclass.
2. Create a class dynamically with `type(name, bases, namespace)`.
3. Write a metaclass with `__new__` or `__init__` to customise class creation.
4. Use `__init_subclass__` to react when a class is subclassed.
5. Use `__set_name__` to let an attribute learn its own name.

**Prerequisites:** Tracks 1 to 3, and Modules 4.1 to 4.3.

---

## Part 1 · Classes Are Objects

Everything in Python is an object, including classes. When you write a `class` statement, Python creates a class object and binds it to the name. You can confirm this: a class has a type, and that type is `type` itself. `type` is the default **metaclass**, the thing that builds classes, just as a class builds instances.

In [ ]:
class Dog:
    pass

d = Dog()

# An instance's type is its class; a class's type is its metaclass.
print("type of an instance:", type(d))        # <class 'Dog'>
print("type of the class:", type(Dog))         # <class 'type'>
print("type of type:", type(type))             # type is its own type

# A class is a first-class object: assign it, store it, pass it.
alias = Dog
print("same class object:", alias is Dog)

## Part 2 · `type` as a Class Factory

Called with one argument, `type(obj)` reports an object's type. Called with **three** arguments, `type(name, bases, namespace)` creates a brand-new class on the spot: a name, a tuple of base classes, and a dictionary of attributes and methods. The familiar `class` statement is essentially syntactic sugar for this call.

In [ ]:
# A normal class statement...
class Point:
    dimensions = 2
    def describe(self):
        return f"{self.dimensions}D point"

# ...is equivalent to building it dynamically with type().
def describe(self):
    return f"{self.dimensions}D point"

DynamicPoint = type(
    "DynamicPoint",          # the class name
    (),                       # base classes (none here)
    {"dimensions": 2, "describe": describe},   # attributes and methods
)

print("from class statement:", Point().describe())
print("from type():", DynamicPoint().describe())
print("both are classes:", type(Point), type(DynamicPoint))

## Part 3 · Writing a Metaclass

A metaclass customises class creation. You define it by subclassing `type` and overriding `__new__` (which builds the class object) or `__init__` (which initialises it). A class then opts in with `class C(metaclass=MyMeta):`. The metaclass code runs **once**, when the class is defined, not when instances are made. This is useful for validating or transforming class definitions.

In [ ]:
class UppercaseAttributes(type):
    """A metaclass that records the class's method names in a registry attribute."""
    def __new__(mcs, name, bases, namespace):
        # Inspect the namespace before the class is built.
        methods = [key for key, value in namespace.items() if callable(value)]
        namespace["method_names"] = methods       # inject a new class attribute
        return super().__new__(mcs, name, bases, namespace)

class Service(metaclass=UppercaseAttributes):
    def start(self): pass
    def stop(self): pass

# The metaclass ran at definition time and added 'method_names'.
print("injected attribute:", Service.method_names)

In [ ]:
# A metaclass can enforce rules on classes, failing early if a definition is wrong.
class RequireDocstring(type):
    def __new__(mcs, name, bases, namespace):
        if name != "Base" and not namespace.get("__doc__"):
            raise TypeError(f"class {name} must have a docstring")
        return super().__new__(mcs, name, bases, namespace)

class Base(metaclass=RequireDocstring):
    pass

class Good(Base):
    """This class is documented."""

print("Good was created:", Good.__name__)

try:
    class Bad(Base):                  # no docstring -> rejected at definition time
        pass
except TypeError as e:
    print("rejected:", e)

## Part 4 · `__init_subclass__` — A Lighter Alternative

Full metaclasses are powerful but heavy, and rarely necessary. The hook `__init_subclass__` lets a base class run code whenever it is subclassed, covering most cases where people once reached for a metaclass: registration, validation, and configuration. It is a class method called automatically on the parent for each new subclass.

In [ ]:
class Plugin:
    registry = []

    def __init_subclass__(cls, **kwargs):
        # Runs automatically each time Plugin is subclassed.
        super().__init_subclass__(**kwargs)
        Plugin.registry.append(cls)
        print(f"registered plugin: {cls.__name__}")

class AudioPlugin(Plugin):
    pass

class VideoPlugin(Plugin):
    pass

print("all plugins:", [cls.__name__ for cls in Plugin.registry])

In [ ]:
# __init_subclass__ can also accept keyword arguments given in the class header,
# which makes for clean, declarative configuration.
class Animal:
    def __init_subclass__(cls, sound="...", **kwargs):
        super().__init_subclass__(**kwargs)
        cls.sound = sound               # configure the subclass from its header

class Dog(Animal, sound="Woof"):
    pass

class Cat(Animal, sound="Meow"):
    pass

print("Dog says", Dog.sound)
print("Cat says", Cat.sound)

## Part 5 · `__set_name__` — Attributes That Know Their Name

When a descriptor or similar object is assigned as a class attribute, `__set_name__(self, owner, name)` is called automatically with the attribute's name. This lets the object learn the name it was bound to without the developer repeating it, which is exactly what made the descriptors in Module 3.4 concise. Here it is shown in isolation.

In [ ]:
class Field:
    """A simple field that remembers the attribute name it was assigned to."""
    def __set_name__(self, owner, name):
        # Called once, when the owning class is being created.
        self.public_name = name
        self.private_name = "_" + name
        print(f"Field bound to attribute '{name}'")

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name, None)

    def __set__(self, instance, value):
        setattr(instance, self.private_name, value)

class User:
    name = Field()        # __set_name__ receives 'name' automatically
    email = Field()       # and 'email' here

u = User()
u.name = "Asha"
u.email = "asha@example.com"
print(u.name, u.email)
print("stored under:", User.name.private_name, User.email.private_name)

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Inspecting class types (Easy)

In [ ]:
class Cat:
    pass

print("instance type:", type(Cat()))
print("class type (metaclass):", type(Cat))
print("is Cat an instance of type?", isinstance(Cat, type))
# Every class is an instance of 'type' unless it uses a custom metaclass.

### Example 2 — Building a class with type() (Easy)

In [ ]:
# Create a class dynamically: name, bases, and a namespace dict.
Greeter = type("Greeter", (), {"hello": lambda self: "hi there"})
g = Greeter()
print(g.hello())
print("class name:", Greeter.__name__)

### Example 3 — A registering base class with __init_subclass__ (Medium)

In [ ]:
class Command:
    commands = {}

    def __init_subclass__(cls, name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if name:
            Command.commands[name] = cls       # register under the given name

class Start(Command, name="start"):
    def run(self): return "starting"

class Stop(Command, name="stop"):
    def run(self): return "stopping"

print("registry:", list(Command.commands))
print("run 'start':", Command.commands["start"]().run())

### Example 4 — A validating metaclass (Medium)

In [ ]:
class MustImplement(type):
    """Ensure subclasses define a 'handle' method."""
    def __new__(mcs, name, bases, namespace):
        if bases and "handle" not in namespace:
            raise TypeError(f"{name} must define handle()")
        return super().__new__(mcs, name, bases, namespace)

class Handler(metaclass=MustImplement):
    pass

class JSONHandler(Handler):
    def handle(self): return "handling JSON"

print(JSONHandler().handle())

try:
    class BrokenHandler(Handler):
        pass                                  # missing handle() -> rejected
except TypeError as e:
    print("rejected:", e)

### Example 5 — A field system with __set_name__ (Difficult)

In [ ]:
class Typed:
    """A descriptor that enforces a type, learning its own name via __set_name__."""
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"expected {self.expected_type.__name__}")
        setattr(instance, self.name, value)

class Product:
    name = Typed(str)
    price = Typed(float)

    def __init__(self, name, price):
        self.name = name
        self.price = price

p = Product("Book", 29.99)
print(p.name, p.price)
try:
    p.price = "free"                          # wrong type -> rejected
except TypeError as e:
    print("rejected:", e)

### Example 6 — A self-registering plugin system (Difficult)

In [ ]:
class Exporter:
    registry = {}

    def __init_subclass__(cls, fmt=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if fmt:
            Exporter.registry[fmt] = cls

    def export(self, data):
        raise NotImplementedError

class CSVExporter(Exporter, fmt="csv"):
    def export(self, data):
        return "csv: " + ",".join(map(str, data))

class JSONExporter(Exporter, fmt="json"):
    def export(self, data):
        return "json: " + str(list(data))

def export_as(fmt, data):
    exporter = Exporter.registry[fmt]()        # look up by format and instantiate
    return exporter.export(data)

print("formats available:", list(Exporter.registry))
print(export_as("csv", [1, 2, 3]))
print(export_as("json", [1, 2, 3]))

---

## Exercises

**Exercise 1 (Easy).** Print the type of the integer `5`, the type of `int`, and confirm with `isinstance` that `int` is an instance of `type`.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use `type()` with three arguments to create a class `Robot` with a single method `beep(self)` that returns `"beep"`. Instantiate it and call the method.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a base class `Shape` whose `__init_subclass__` appends each subclass to a class-level list `Shape.kinds`. Define two subclasses and print the collected list of names.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a metaclass `NoUnderscoreMethods` that raises a `TypeError` if any subclass defines a method whose name starts with a single underscore (ignore dunder names). Test it with a valid and an invalid class.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a descriptor `Positive` that uses `__set_name__` to learn its attribute name and rejects non-positive numbers. Use it for an `amount` attribute on an `Order` class and demonstrate rejection.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
print("type of 5:", type(5))
print("type of int:", type(int))
print("int is an instance of type:", isinstance(int, type))

**Solution 2**

In [ ]:
Robot = type("Robot", (), {"beep": lambda self: "beep"})
print(Robot().beep())

**Solution 3**

In [ ]:
class Shape:
    kinds = []
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Shape.kinds.append(cls.__name__)

class Circle(Shape): pass
class Square(Shape): pass

print(Shape.kinds)

**Solution 4**

In [ ]:
class NoUnderscoreMethods(type):
    def __new__(mcs, name, bases, namespace):
        for key, value in namespace.items():
            if callable(value) and key.startswith("_") and not key.startswith("__"):
                raise TypeError(f"{name}.{key} uses a leading underscore")
        return super().__new__(mcs, name, bases, namespace)

class Good(metaclass=NoUnderscoreMethods):
    def run(self): pass

print("Good created:", Good.__name__)

try:
    class Bad(metaclass=NoUnderscoreMethods):
        def _helper(self): pass
except TypeError as e:
    print("rejected:", e)

**Solution 5**

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = "_" + name
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.name)
    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError("must be positive")
        setattr(instance, self.name, value)

class Order:
    amount = Positive()
    def __init__(self, amount):
        self.amount = amount

o = Order(10)
print("amount:", o.amount)
try:
    o.amount = -5
except ValueError as e:
    print("rejected:", e)

---

## Summary and Key Points

- Classes are objects; an instance's type is its class, and a class's type is its metaclass, which is `type` by default.
- `type(name, bases, namespace)` creates a class dynamically; the `class` statement is sugar for this call.
- A metaclass subclasses `type` and overrides `__new__` or `__init__` to customise or validate class creation; its code runs once, at class definition.
- `__init_subclass__` is a lighter hook that runs when a class is subclassed, handling registration, validation, and configuration (including keyword arguments in the class header) without a full metaclass.
- `__set_name__(self, owner, name)` lets a class attribute learn its own name at class-creation time, which is what makes descriptors concise.

### Next module: 4.5 — Introspection & Reflection

The next module covers examining objects at runtime: `dir`, `getattr` and friends, the `inspect` module, and working with annotations.